# 03. Térbeli Elemzés és Interaktív Térképek

**Cél**: A kőbányai ingatlanok térbeli mintázatának, lokációs ársűrűségének és hálózati elérhetőségi viszonyainak térinformatikai vizsgálata.

**Adatalap és mintaméretek**:
- A teljes garantált pontos geokódolású GIS adatbázis **N = 296 db** ingatlan (254 db eladó + 42 db kiadó lakás).
- Jelen eladási térbeli elemzés a garantált pontos **eladó lakások mintáján (N = 254 db)** fut, biztosítva a módszertani konzisztenciát a 04. hedonikus és a 10. Moran modellekkel.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
df_pontos = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].copy()
print(f"Garantált pontos koordinátákkal rendelkező eladó hirdetések száma: {len(df_pontos)} db.")

### 1. Térbeli Mintavétel és Elérhetőségi KPI-k
A reprezentatív pontos eladói minta és az infrastrukturális csomópontok távolsági mutatói.

In [ ]:
n_pontos = len(df_pontos)
in_mazsa_10p = int((df_pontos['mazsa_10p_seta'] == 1).sum()) if 'mazsa_10p_seta' in df_pontos.columns else 0
in_metro_10p = int((df_pontos['metro_10p_seta'] == 1).sum()) if 'metro_10p_seta' in df_pontos.columns else 0
atlag_belvaros_km = df_pontos['tavolsag_belvaros_halozati_m'].mean() / 1000 if 'tavolsag_belvaros_halozati_m' in df_pontos.columns else 6.5
med_nm_ar_pontos = df_pontos['nm_ar_huf'].median()

kpi_cards = [
    ("Garantált Pontos Minta", f"{n_pontos} db", "100% valós koordináta (eladó)", "#1e3a8a"),
    ("Mázsa tér 10p Séta", f"{in_mazsa_10p} db", f"{in_mazsa_10p/n_pontos*100:.1f}% lefedettség", "#ec4899"),
    ("Metró 10p Séta", f"{in_metro_10p} db", f"{in_metro_10p/n_pontos*100:.1f}% lefedettség", "#2563eb"),
    ("Átlag Táv Belváros", f"{atlag_belvaros_km:.2f} km", "Közúti/gyalogos hálózat", "#059669"),
    ("Pontos Minta Medián Ár", fmt_huf(med_nm_ar_pontos), "Fajlagos eladási ár/m²", "#7c3aed"),
    ("Városrészek Száma", f"{df_pontos['varosresz'].nunique()} db", "Térbeli lefedettség", "#d97706")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Valós Térképi Pontmegjelenítés Négyzetméterár Szerint
Az összes garantált pontos ingatlan geokódolt pontként valós térképi alapon (`scatter_map`), színskálával jelölve a fajlagos árat.

In [ ]:
fig1 = px.scatter_map(
    df_pontos,
    lat='geokodolt_lat',
    lon='geokodolt_lon',
    color='nm_ar_huf',
    size='alapterulet_nm',
    hover_name='cim_teljes',
    hover_data={'nm_ar_huf': ':.0f', 'ar_millio_ft': ':.1f', 'alapterulet_nm': True, 'szobaszam_osszes': True, 'varosresz': True},
    color_continuous_scale='Viridis',
    range_color=[df_pontos['nm_ar_huf'].quantile(0.05), df_pontos['nm_ar_huf'].quantile(0.95)],
    zoom=12.2,
    center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
    map_style='carto-positron',
    title='Kőbányai lakások térbeli elhelyezkedése és négyzetméterára (Carto Positron térképen)'
)
fig1.update_layout(height=550, margin={"r":0,"t":40,"l":0,"b":0})
fig1.show()

### 3. Ársűrűségi Hőtérkép és Mázsa Téri Távolsági Gradiens
Hol koncentrálódnak a magas fajlagos árak, és hogyan függ az ár a központi területektől való hálózati távolságtól?

In [ ]:
fig2 = px.density_map(
    df_pontos,
    lat='geokodolt_lat',
    lon='geokodolt_lon',
    z='nm_ar_huf',
    radius=22,
    zoom=12.2,
    center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
    map_style='carto-positron',
    title='Fajlagos ársűrűségi hőtérkép (Density Map) Kőbányán'
)
fig2.update_layout(height=520, margin={"r":0,"t":40,"l":0,"b":0})
fig2.show()

# Hálózati távolság a Mázsa tértől vs Ár/m²
if 'tavolsag_mazsa_halozati_m' in df_pontos.columns:
    fig3 = px.scatter(
        df_pontos,
        x='tavolsag_mazsa_halozati_m',
        y='nm_ar_huf',
        color='varosresz',
        trendline='lowess',
        title='Hálózati séta távolság a Mázsa tértől (m) vs. Négyzetméterár (LOWESS trendvonallal)',
        labels={'tavolsag_mazsa_halozati_m': 'Mázsa tér hálózati távolság (méter)', 'nm_ar_huf': 'Ár / m² (HUF)'},
        template=PLOTLY_TEMPLATE
    )
    fig3.update_layout(height=480)
    fig3.show()

### 4. Interaktív Térképi Szűrő és Rétegkezelő Pult
Dinamikusan szűrhető térkép hirdetéstípus, árkategória és színezési változó alapján.

In [ ]:
w_szin = widgets.Dropdown(
    options=[('Négyzetméterár', 'nm_ar_huf'), ('Alapterület', 'alapterulet_nm'), ('Szobaszám', 'szobaszam_osszes'), ('Állapot Kód', 'allapot_kod')],
    value='nm_ar_huf',
    description='Színezés:'
)
w_meret = widgets.FloatSlider(min=4, max=15, step=1, value=7, description='Pontméret:')
w_tipus = widgets.RadioButtons(options=['Mindkettő', 'eladó', 'kiadó'], value='eladó', description='Típus:')

out_map = widgets.Output()

def frissit_terkep(*args):
    sub = df[df['minta_garantalt_pontos'] == 1].copy()
    if w_tipus.value != 'Mindkettő':
        mapped_val = 'elado' if w_tipus.value == 'eladó' else 'kiado'
        sub = sub[sub['listing_type'] == mapped_val]
        
    with out_map:
        clear_output(wait=True)
        fig = px.scatter_map(
            sub,
            lat='geokodolt_lat',
            lon='geokodolt_lon',
            color=w_szin.value,
            size_max=w_meret.value,
            hover_name='cim_teljes',
            zoom=12.2,
            center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
            map_style='carto-positron',
            title=f'Interaktív térkép: {w_szin.label} alapján (N={len(sub)})'
        )
        fig.update_layout(height=500, margin={"r":0,"t":40,"l":0,"b":0})
        fig.show()

w_szin.observe(frissit_terkep, names='value')
w_meret.observe(frissit_terkep, names='value')
w_tipus.observe(frissit_terkep, names='value')

display(widgets.HBox([w_szin, w_meret, w_tipus]))
display(out_map)
frissit_terkep()